In [0]:
dbutils.widgets.text("p_data_source", "")

In [0]:
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
dbutils.widgets.text("p_file_date", "")

In [0]:
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../Includes/configs" 

In [0]:
%run "../SetUp/setup"

### Access Azure Data Lake using Service Principal
**Steps to follow:**
1. Register Azure AD Application/ Service Principal
2. Generate a secret/ password for the application
3. Set spark config with App/ Client Id, Directory/ Tenant Id & Secret
4. Assign role "Storage Blob Data Contributor" to the Data Lake

path,name,size,modificationTime
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-21/,2021-03-21/,0,1768733801000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-28/,2021-03-28/,0,1768733591000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-04-18/,2021-04-18/,0,1768733721000


[FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/__unitystorage/', name='__unitystorage/', size=0, modificationTime=1769227742000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/drivers_convert_to_delta/', name='drivers_convert_to_delta/', size=0, modificationTime=1769359802000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/drivers_convert_to_delta_new/', name='drivers_convert_to_delta_new/', size=0, modificationTime=1769360049000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/results_external/', name='results_external/', size=0, modificationTime=1769228062000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/results_partitioned/', name='results_partitioned/', size=0, modificationTime=1769228706000)]

In [0]:
%run "../Includes/comm_func" 

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

In [0]:
results_schema = StructType(fields=[StructField("resultId", IntegerType(), False),
                                    StructField("raceId", IntegerType(), True),
                                    StructField("driverId", IntegerType(), True),
                                    StructField("constructorId", IntegerType(), True),
                                    StructField("number", IntegerType(), True),
                                    StructField("grid", IntegerType(), True),
                                    StructField("position", IntegerType(), True),
                                    StructField("positionText", StringType(), True),
                                    StructField("positionOrder", IntegerType(), True),
                                    StructField("points", FloatType(), True),
                                    StructField("laps", IntegerType(), True),
                                    StructField("time", StringType(), True),
                                    StructField("milliseconds", IntegerType(), True),
                                    StructField("fastestLap", IntegerType(), True),
                                    StructField("rank", IntegerType(), True),
                                    StructField("fastestLapTime", StringType(), True),
                                    StructField("fastestLapSpeed", FloatType(), True),
                                    StructField("statusId", StringType(), True)])

In [0]:
results_df = spark.read.schema(results_schema).json(f"{raw_folder_path}/{v_file_date}/results.json")

In [0]:
#display(results_df)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
results_with_column_df = results_df.withColumnRenamed("resultId", "result_id").withColumnRenamed("raceId", "race_id").withColumnRenamed("driverId", "driver_id").withColumnRenamed("constructorId", "constructor_id").withColumnRenamed("positionText", "position_text").withColumnRenamed("positionOrder", "position_order").withColumnRenamed("fastestLap", "fastest_lap").withColumnRenamed("fastestLapTime", "fastest_lap_time").withColumnRenamed("fastestLapSpeed", "fastest_lap_speed").withColumn("ingestion_date", current_timestamp()).withColumn("data_source", lit(v_data_source)).withColumn("file_date", lit(v_file_date))

In [0]:
results_final_df = results_with_column_df.drop("statusId")

In [0]:
results_final_df = results_final_df.dropDuplicates(
    ["race_id", "driver_id"]
)

In [0]:
results_final_df.write \
    .mode("append") \
    .format("delta") \
    .partitionBy("race_id") \
    .saveAsTable("f1_processed.results")

In [0]:
%sql
SELECT * FROM f1_processed.results;

result_id,race_id,driver_id,constructor_id,number,grid,position,position_text,position_order,points,laps,time,milliseconds,fastest_lap,rank,fastest_lap_time,fastest_lap_speed,ingestion_date,data_source,file_date
20677,351,1,1,2,3,null,R,19,0.0,35,\N,null,31,10,1:50.750,164.901,2026-01-26T15:22:00.779654Z,Ergast API,2021-03-21
20676,351,2,15,22,14,null,R,18,0.0,36,\N,null,29,16,1:52.475,162.372,2026-01-26T15:22:00.779654Z,Ergast API,2021-03-21
20663,351,3,131,4,7,5,5,5,10.0,61,+49.394,7122973,55,8,1:50.125,165.837,2026-01-26T15:22:00.779654Z,Ergast API,2021-03-21
20659,351,4,6,8,1,1,1,1,25.0,61,1:57:53.579,7073579,58,1,1:47.976,169.137,2026-01-26T15:22:00.779654Z,Ergast API,2021-03-21
20674,351,5,205,19,19,16,16,16,0.0,58,\N,null,53,18,1:53.051,161.544,2026-01-26T15:22:00.779654Z,Ergast API,2021-03-21
20665,351,9,4,11,8,7,7,7,6.0,61,+1:26.559,7160138,56,3,1:49.255,167.157,2026-01-26T15:22:00.779654Z,Ergast API,2021-03-21
20675,351,10,166,24,18,null,R,17,0.0,49,\N,null,38,19,1:53.559,160.822,2026-01-26T15:22:00.779654Z,Ergast API,2021-03-21
20666,351,13,6,7,24,8,8,8,4.0,61,+1:53.297,7186876,45,12,1:52.079,162.945,2026-01-26T15:22:00.779654Z,Ergast API,2021-03-21
20681,351,15,205,18,21,null,R,23,0.0,27,\N,null,20,21,1:56.386,156.915,2026-01-26T15:22:00.779654Z,Ergast API,2021-03-21
20667,351,16,10,14,15,9,9,9,2.0,61,+2:02.416,7195995,45,15,1:52.473,162.374,2026-01-26T15:22:00.779654Z,Ergast API,2021-03-21


In [0]:
%sql
SELECT COUNT(1) FROM f1_processed.results
WHERE file_date = '2021-03-21'

count(1)
24869


In [0]:
%sql
SELECT race_id, driver_id, COUNT(1) FROM f1_processed.results
GROUP BY race_id, driver_id
HAVING COUNT(1) > 1
ORDER BY race_id, driver_id DESC;

race_id,driver_id,count(1)
